# Fake-data velocity maps: direct mean vs likelihood fit

Uses Alex's `dipole.generate_fake_data` to make a mock halo with a known reflex vector, then
1. compares **direct** v_travel (−mean Cartesian velocity) with the **fit** (`dipole.lnlike`),
2. plots sky scatter maps of the velocities in **spherical** (v_r, v_θ, v_φ) and **Cartesian** (v_x, v_y, v_z) components, laid out like the ⟨v_R⟩, ⟨v_θ⟩, ⟨v_φ⟩ maps in Emily's paper,
3. tests whether a **different centre definition** (position centre or velocity zero point) changes either estimate.

**How the fake data are built** (`generate_fake_data`): positions are a Gaussian ball (σ = 50 kpc) around the origin; each star's *intrinsic* spherical velocity is drawn from a Gaussian with mean `MEAN` and dispersions `SIGMA`; then the reflex is added as `v_obs = v_intrinsic − R`, so the halo appears to move with −R. Both methods should return R.

In [ ]:
import os, sys, types
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize

sys.path.insert(0, os.path.abspath('../code'))
try:
    import dipole
except ImportError:
    # off SCC: stand-ins for packages the fake-data functions never use
    for name in ['gizmo_analysis', 'halo_analysis', 'utilities', 'utilities.coordinate', 'pygaia']:
        sys.modules.setdefault(name, types.ModuleType(name))
    sys.modules['utilities'].coordinate = sys.modules['utilities.coordinate']
    sys.modules['pygaia'].proper_motion_uncertainty = None
    import dipole

## Parameters — edit here

In [ ]:
N = 1000
SEED = 1
R_TRUE = np.array([20., -10., 25.])    # reflex vector (Cartesian, km/s): velocity of the centre relative to the halo
MEAN = np.array([0., 0., 0.])           # intrinsic mean (v_r, v_theta, v_phi); try [0, 0, 50] for a rotating halo
SIGMA = np.array([100., 80., 80.])      # intrinsic dispersions (sigma_r, sigma_theta, sigma_phi)

PARS = np.r_[R_TRUE, MEAN, np.log10(SIGMA)]

# plotting
CMAP = 'viridis'
POINT_SIZE = 8
FLIP_LON = False      # True puts longitude increasing to the left (sky convention)

## Helpers

In [ ]:
def sph_basis(xyz):
    """Unit vectors r_hat, theta_hat, phi_hat at each position (rows = stars)."""
    x, y, z = xyz.T
    r = np.linalg.norm(xyz, axis=1)
    th, ph = np.arccos(z / r), np.arctan2(y, x)
    r_hat = xyz / r[:, None]
    th_hat = np.stack([np.cos(th) * np.cos(ph), np.cos(th) * np.sin(ph), -np.sin(th)], axis=1)
    ph_hat = np.stack([-np.sin(ph), np.cos(ph), np.zeros_like(ph)], axis=1)
    return r_hat, th_hat, ph_hat


def sph_to_xyz(v_sph, xyz):
    """Inverse of dipole.vel_xyz_to_sph_optimized."""
    r_hat, th_hat, ph_hat = sph_basis(xyz)
    return v_sph[:, :1] * r_hat + v_sph[:, 1:2] * th_hat + v_sph[:, 2:3] * ph_hat


def direct_v_travel(v_xyz):
    """-(mean Cartesian velocity). Uses no positions at all."""
    return -v_xyz.mean(axis=0)


def fit_v_travel(xyz, v_sph):
    """Alex's likelihood, SLSQP on the per-star mean -lnL (same as scc/03_fit_m12i.py)."""
    n = len(xyz)
    initial = [0, 0, 0, 0, 0, 0, np.log10(50), np.log10(50), np.log10(50)]
    soln = minimize(lambda p: -dipole.lnlike(p, xyz, v_sph) / n, initial, bounds=dipole.get_bounds(),
                    method='SLSQP', options={'maxiter': 500, 'ftol': 1e-10})
    return soln.x


def describe(vec):
    v = np.linalg.norm(vec)
    return f'({vec[0]:6.1f}, {vec[1]:6.1f}, {vec[2]:6.1f})  |v| = {v:5.1f} km/s'


def sky_angles(xyz):
    """Longitude (azimuth phi) and latitude (90 deg - theta) in radians, for a Mollweide plot."""
    x, y, z = xyz.T
    lon = np.arctan2(y, x)
    lat = np.arcsin(z / np.linalg.norm(xyz, axis=1))
    return (-lon if FLIP_LON else lon), lat


def velocity_maps(xyz, v_xyz, title, vlim, mark=None):
    """2 x 3 Mollweide scatter maps: top = spherical components, bottom = Cartesian."""
    v_sph = dipole.vel_xyz_to_sph_optimized(v_xyz, xyz)
    lon, lat = sky_angles(xyz)
    rows = [(v_sph, [r'$v_r$', r'$v_\theta$', r'$v_\phi$']),
            (v_xyz, [r'$v_x$', r'$v_y$', r'$v_z$'])]
    fig, axes = plt.subplots(2, 3, figsize=(15, 6.2), subplot_kw={'projection': 'mollweide'})
    for (vals, labels), axrow in zip(rows, axes):
        for k, ax in enumerate(axrow):
            sc = ax.scatter(lon, lat, c=vals[:, k], s=POINT_SIZE, cmap=CMAP, vmin=-vlim, vmax=vlim)
            if mark is not None:
                mlon, mlat = sky_angles(np.atleast_2d(mark))
                ax.plot(mlon, mlat, marker='*', ms=16, color='k', mec='w')
            ax.set_title(labels[k], fontsize=13)
            ax.set_xticklabels([]); ax.set_yticklabels([])
            ax.grid(alpha=0.3)
            fig.colorbar(sc, ax=ax, orientation='horizontal', fraction=0.05, pad=0.04, label='km/s')
    fig.suptitle(title, fontsize=13)
    fig.tight_layout(h_pad=3)
    return fig

## 1. Generate fake data and compare the two estimates

In [ ]:
xyz, v_sph = dipole.generate_fake_data(PARS, size=(N, 3), seed=SEED)
v_xyz = sph_to_xyz(v_sph, xyz)
assert np.allclose(dipole.vel_xyz_to_sph_optimized(v_xyz, xyz), v_sph)   # round trip check

r_direct = direct_v_travel(v_xyz)
p_fit = fit_v_travel(xyz, v_sph)
noise = SIGMA.mean() / np.sqrt(N)

print(f'N = {N}, expected shot noise per component ~ {noise:.1f} km/s\n')
print('true   ', describe(R_TRUE))
print('direct ', describe(r_direct))
print('fit    ', describe(p_fit[:3]))
print('\nfit mean (v_r, v_theta, v_phi):', np.round(p_fit[3:6], 1), '  true', MEAN)
print('fit sigma                    :', np.round(10**p_fit[6:], 1), '  true', SIGMA)

## 2. Velocity maps

The star marks the direction of R (where the centre is heading).

**Reflex motion** (every star moves with −R, no intrinsic motion): in Cartesian components it is a *uniform* offset, (V_x, V_y, V_z) = −R for every star (printed under each row); in spherical components the same uniform vector turns into sky patterns, a dipole in v_r and position-dependent v_θ, v_φ. This is the signal Emily's maps show.

Rows: the true R, then the direct and fitted estimates from the N = 1000 fake data above, on the same colour scale.


In [ ]:
# reflex motion for the true R and for the two estimates from the N = 1000 fake data (section 1)
CASES = [('true reflex', R_TRUE), (f'direct estimate (N = {N})', r_direct), (f'fit estimate (N = {N})', p_fit[:3])]
lon, lat = sky_angles(xyz)
vlim = np.linalg.norm(R_TRUE)          # same colour scale in every row

fig, axes = plt.subplots(len(CASES), 3, figsize=(15, 12.5), subplot_kw={'projection': 'mollweide'})
fig.subplots_adjust(left=0.02, right=0.98, top=0.95, bottom=0.08, wspace=0.08, hspace=0.85)
for axrow, (name, R) in zip(axes, CASES):
    v_sph_reflex = dipole.vel_xyz_to_sph_optimized(np.broadcast_to(-R, xyz.shape), xyz)
    mlon, mlat = sky_angles(np.atleast_2d(R))
    for k, (ax, label) in enumerate(zip(axrow, [r'$v_r$', r'$v_\theta$', r'$v_\phi$'])):
        sc = ax.scatter(lon, lat, c=v_sph_reflex[:, k], s=POINT_SIZE, cmap=CMAP, vmin=-vlim, vmax=vlim)
        ax.plot(mlon, mlat, marker='*', ms=16, color='k', mec='w')
        ax.set_title(label, fontsize=13)
        ax.set_xticklabels([]); ax.set_yticklabels([])
        ax.grid(alpha=0.3)
        cb = fig.colorbar(sc, ax=ax, orientation='horizontal', fraction=0.05, pad=0.04, label='km/s')
    # row title above, reflex vector below (positions taken from the drawn axes)
    top = axrow[1].get_position().y1
    bottom = cb.ax.get_position().y0
    fig.text(0.5, top + 0.03, f'reflex motion: {name}', ha='center', va='bottom', fontsize=14)
    vx, vy, vz = -R                    # the velocity every star is given (= -R)
    fig.text(0.5, bottom - 0.055, rf'$(V_x,\ V_y,\ V_z) = ({vx:.1f},\ {vy:.1f},\ {vz:.1f})$ km/s',
             ha='center', va='top', fontsize=13)

**Observed fake data** (intrinsic motion + reflex). With N = 1000 and σ ≈ 90 km/s per star, the ~30 km/s reflex is buried in single-star scatter; Emily's maps are *means* over many stars per pixel. Raise N or lower SIGMA to see the pattern emerge.

In [ ]:
fig = velocity_maps(xyz, v_xyz, f'fake data, N = {N}', vlim=150, mark=R_TRUE)

## 3. Does a different centre definition matter?

Two different things can be called "the centre":

- **Velocity zero point**: the velocity subtracted from all stars. Adding a constant u to every Cartesian velocity shifts *both* estimates by exactly −u: the direct mean trivially, and the fit because the model only ever uses v + R.
- **Position centre**: the origin used to compute r̂, θ̂, φ̂. The direct mean never uses positions, so it is unaffected. The fit does, so it can change.

In [ ]:
u = np.array([10., 0., 0.])
r_d = direct_v_travel(v_xyz + u)
r_f = fit_v_travel(xyz, dipole.vel_xyz_to_sph_optimized(v_xyz + u, xyz))[:3]
print('velocity zero point shifted by u =', u)
print('  direct change:', np.round(r_d - r_direct, 2))
print('  fit change   :', np.round(r_f - p_fit[:3], 2))

In [ ]:
OFFSETS = [0, 2, 5, 10, 20, 50]      # kpc, along x
MEANS = {'no rotation': [0, 0, 0], 'rotating halo, v_phi = 50': [0, 0, 50]}

fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
for ax, (label, mean) in zip(axes, MEANS.items()):
    pars = np.r_[R_TRUE, mean, np.log10(SIGMA)]
    x0, vs0 = dipole.generate_fake_data(pars, size=(N, 3), seed=SEED)
    vx0 = sph_to_xyz(vs0, x0)
    d = direct_v_travel(vx0)                      # same for every offset
    fits = []
    for off in OFFSETS:
        x_assumed = x0 - np.array([off, 0, 0])   # positions measured from the wrong centre
        fits.append(fit_v_travel(x_assumed, dipole.vel_xyz_to_sph_optimized(vx0, x_assumed))[:3])
    fits = np.array(fits)
    for k, c in enumerate(['C0', 'C1', 'C2']):
        ax.plot(OFFSETS, fits[:, k], 'o-', color=c, label=f'fit R_{"xyz"[k]}')
        ax.axhline(d[k], color=c, ls=':', lw=1)
        ax.axhline(R_TRUE[k], color=c, ls='--', lw=0.8, alpha=0.5)
    ax.set_title(label)
    ax.set_xlabel('position-centre offset along x [kpc]')
axes[0].set_ylabel('km/s')
axes[0].plot([], [], 'k:', label='direct (no position dependence)')
axes[0].plot([], [], 'k--', alpha=0.5, label='true')
axes[0].legend(fontsize=8, frameon=False)
fig.tight_layout()

What to look for: with no rotation the fit barely moves; with a rotating halo, an offset centre makes the rotation look like a bulk flow on one side of the sky and the fit's R_y drifts (for an offset along x). The direct mean stays put.

Maps of the rotating halo seen from the wrong centre:

In [ ]:
OFF = 20
pars = np.r_[R_TRUE, [0, 0, 50], np.log10(SIGMA)]
x0, vs0 = dipole.generate_fake_data(pars, size=(N, 3), seed=SEED)
vx0 = sph_to_xyz(vs0, x0)
fig = velocity_maps(x0, vx0, 'rotating halo, true centre', vlim=150, mark=R_TRUE)
fig = velocity_maps(x0 - [OFF, 0, 0], vx0, f'rotating halo, centre offset by {OFF} kpc along x', vlim=150, mark=R_TRUE)